# MediQ T5-Small Report Summarization — Google Colab GPU Training Notebook

This notebook orchestrates the fine-tuning of **T5-small** on the PubMed scientific papers dataset (`article` $\rightarrow$ `abstract`) with **persistent Google Drive storage**.

> **CRITICAL SAFETY GUARANTEE**:
> All training checkpoints, evaluation metrics, and final model exports are saved **directly to Google Drive**.
> Nothing is saved solely under `/content/`, ensuring zero progress loss if the Colab runtime disconnects.

> **CLINICAL LIMITATION NOTICE**:
> The training dataset consists of scientific biomedical literature articles and author abstracts (`armanc/scientific_papers`, PubMed configuration).
> It serves as a technical proxy for biomedical domain text summarization. It is **NOT** a clinical patient-report dataset and is **NOT** clinically validated for patient care or medical diagnosis.

### Step 1: Install Required Dependencies
Install Hugging Face Transformers, Datasets, Evaluate, ROUGE Score, Accelerate, and SentencePiece.

In [ ]:
!pip install --upgrade pip
!pip install transformers datasets evaluate rouge_score accelerate sentencepiece torch

### Step 2: Mount Google Drive
Mount persistent Google Drive storage **BEFORE** any training or data copying begins.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Google Drive mounted successfully.")

### Step 3: Verify Drive Path and Write Access
Confirm that Google Drive is accessible and writable. Training will abort if write access fails.

In [ ]:
import os
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/MediQ/training/report")
test_file = Path("/content/drive/MyDrive/.mediq_drive_write_test")

try:
    test_file.write_text("write_test_ok", encoding="utf-8")
    test_file.unlink()
    print(f"Google Drive write access verified.")
except Exception as exc:
    raise RuntimeError(f"Cannot write to Google Drive: {exc}. Please verify Drive permissions before proceeding!")

### Step 4: Define Project Paths & Create Persistent Directories
Prepare the persistent directory layout on Google Drive.

In [ ]:
RUNS_DIR = DRIVE_ROOT / "runs"
CHECKPOINTS_DIR = DRIVE_ROOT / "checkpoints"
EVALUATION_DIR = DRIVE_ROOT / "evaluation"
EXPORTS_DIR = DRIVE_ROOT / "exports" / "t5_small_summarizer"

for d in [RUNS_DIR, CHECKPOINTS_DIR, EVALUATION_DIR, EXPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)
    print(f"Persistent directory ready: {d}")

### Step 5: Validate Preprocessed Dataset & Tokenizer (Preflight Check)
Run the preflight validation check. This confirms that all JSONL files exist, data fields are intact, and the tokenizer loads properly without starting training.

In [ ]:
# Set DATA_DIR to the location of your processed dataset
# Either in the cloned repo (e.g. MediQ/backend/training/report/processed) or on Drive
DATA_DIR = Path("/content/drive/MyDrive/MediQ/training/report/processed")
if not DATA_DIR.exists():
    DATA_DIR = Path("/content/MediQ/backend/training/report/processed")

print(f"Using processed data directory: {DATA_DIR}")

!python train.py \
    --data_dir "{DATA_DIR}" \
    --google_drive \
    --drive_root "{DRIVE_ROOT}" \
    --validate_only

### Step 6: Execute T5-Small Training on GPU
Start fine-tuning T5-small. Checkpoints and logs are saved directly to Google Drive.

In [ ]:
!python train.py \
    --data_dir "{DATA_DIR}" \
    --google_drive \
    --drive_root "{DRIVE_ROOT}" \
    --model_name t5-small \
    --epochs 3 \
    --batch_size 8 \
    --gradient_accumulation_steps 4 \
    --learning_rate 3e-4 \
    --max_source_length 512 \
    --max_target_length 128 \
    --save_period 1 \
    --fp16

### Step 7: Evaluate Model on Held-Out Splits (ROUGE-1, ROUGE-2, ROUGE-L)
Run evaluation on the held-out validation and test splits and save metrics to Google Drive.

In [ ]:
# Evaluate on test split
!python evaluate.py \
    --checkpoint "{EXPORTS_DIR}" \
    --data_dir "{DATA_DIR}" \
    --split test \
    --google_drive \
    --drive_root "{DRIVE_ROOT}"

# Display saved evaluation metrics
import json
metrics_file = EVALUATION_DIR / "eval_test_metrics.json"
if metrics_file.exists():
    print(json.dumps(json.loads(metrics_file.read_text()), indent=2))

### Step 8: Test Summary Generation
Generate a sample abstractive summary using the trained checkpoint.

In [ ]:
sample_text = """A prospective clinical cohort study investigated the therapeutic efficacy and safety of dietary 
supplementation combined with regular exercise in adult patients diagnosed with mild metabolic syndrome. 
Over a 12-month follow-up period, participants demonstrated statistically significant reductions in fasting blood 
glucose levels and systolic blood pressure compared to the baseline control cohort."""

!python infer.py \
    --checkpoint "{EXPORTS_DIR}" \
    --text "{sample_text}"

### Step 9: Export Model Artifacts for Backend Integration
Export the fine-tuned model and metadata for deployment to the MediQ FastAPI backend.

In [ ]:
!python export.py \
    --checkpoint "{EXPORTS_DIR}" \
    --google_drive \
    --drive_root "{DRIVE_ROOT}"